# Staging — DuckDB Native

Replaces the pandas row-by-row CSV reader with DuckDB's `read_csv` which is
orders of magnitude faster.

- **Resumable** — skips files already staged (checks for existing parquet)
- **No destructive reset** — never wipes the stage folder
- **Controllable** — set `MAX_FILES` to limit how many files are processed

In [1]:
## Config

import duckdb
import os
from pathlib import Path
import time

RAW_DIR   = Path("/home/jovyan/data/sim/raw/sim_test_decoded/2025/")
STAGE_DIR = Path("/home/jovyan/data/stage/handover_events")
TMP_DIR   = "/home/jovyan/data/tmp"

# Column positions (zero-based) in the raw CSV
# id, test_type, creation_time, collection_time, insert_time, sim_id, ...
COL_EVENT_TS   = '03'   # collection_time
COL_VEHICLE_ID = '05'   # sim_id
COL_IMSI       = '13' # imsi
COL_CELL_ID    = '18' # cell_id
COL_RAT        = '23' # technology (4G/5G)


# Set to an integer to limit files processed (useful for testing)
# Set to None to process all files
MAX_FILES = None

STAGE_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

print(f"RAW_DIR:   {RAW_DIR}")
print(f"STAGE_DIR: {STAGE_DIR}")
print(f"MAX_FILES: {MAX_FILES}")

RAW_DIR:   /home/jovyan/data/sim/raw/sim_test_decoded/2025
STAGE_DIR: /home/jovyan/data/stage/handover_events
MAX_FILES: None


In [2]:
## Connect DuckDB

con = duckdb.connect()  # in-memory — staging doesn't need persistence
con.execute(f"SET temp_directory='{TMP_DIR}'")
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=8")

import os, psutil
print(f"CPUs: {os.cpu_count()}")
print(f"RAM:  {round(psutil.virtual_memory().total / (1024**3), 1)} GB")

CPUs: 24
RAM:  39.0 GB


In [3]:
## Find raw files

exts = {".gz", ".csv", ".txt"}
all_files = sorted([
    p for p in RAW_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in exts
])

raw_files = all_files[:MAX_FILES] if MAX_FILES else all_files

print(f"Total files found: {len(all_files)}")
print(f"Files to consider: {len(raw_files)}")

Total files found: 304
Files to consider: 304


In [4]:
## Process files — resumable, DuckDB native

skipped = 0
processed = 0
total_good = 0
total_bad = 0
run_start = time.time()

for i, path in enumerate(raw_files, 1):

    # Derive the expected output path from the filename date
    # e.g. 2025_06_15.txt -> event_date=2025-06-15
    stem = path.stem  # e.g. '2025_06_15'
    event_date = stem.replace('_', '-')  # e.g. '2025-06-15'
    out_dir  = STAGE_DIR / f"event_date={event_date}"
    out_file = out_dir / "stage.parquet"

    # --- Resumable: skip if already staged ---
    if out_file.exists():
        skipped += 1
        print(f"[{i}/{len(raw_files)}] SKIP {path.name}")
        continue

    file_start = time.time()

    try:
        out_dir.mkdir(parents=True, exist_ok=True)

        # DuckDB reads the CSV and writes parquet in one shot
        # column0..columnN maps to zero-based column positions
        result = con.execute(f"""
            COPY (
                SELECT
                    column{COL_VEHICLE_ID}  AS vehicle_id,
                    column{COL_IMSI}        AS imsi,
                    column{COL_EVENT_TS}    AS event_ts,
                    column{COL_CELL_ID}     AS cell_id,
                    column{COL_RAT}         AS rat,
                    '{path.name}'           AS source_file
                FROM read_csv(
                    '{path}',
                    header        = false,
                    ignore_errors = true,
                    compression   = 'auto',
                    delim         = ','
                )
                WHERE column{COL_VEHICLE_ID} IS NOT NULL
                  AND column{COL_CELL_ID}    IS NOT NULL
                  AND column{COL_EVENT_TS}   IS NOT NULL
            ) TO '{out_file}' (FORMAT PARQUET)
        """)

        elapsed = time.time() - file_start
        # Count rows written
        n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_file}')").fetchone()[0]
        total_good += n_rows
        processed += 1
        print(f"[{i}/{len(raw_files)}] {path.name} → {n_rows:,} rows ({elapsed:.1f}s)")

    except Exception as e:
        total_bad += 1
        print(f"[{i}/{len(raw_files)}] ERROR {path.name}: {e}")

total_elapsed = time.time() - run_start
print(f"\nDone. {processed} processed, {skipped} skipped, {total_bad} errors")
print(f"Total rows staged: {total_good:,}")
print(f"Total runtime: {total_elapsed/60:.1f} min")

[1/304] SKIP 2025_01_01.gz
[2/304] SKIP 2025_01_02.gz
[3/304] SKIP 2025_01_03.gz
[4/304] SKIP 2025_01_04.gz
[5/304] SKIP 2025_01_05.gz
[6/304] SKIP 2025_01_06.gz
[7/304] SKIP 2025_01_07.gz
[8/304] SKIP 2025_01_08.gz
[9/304] SKIP 2025_01_09.gz
[10/304] SKIP 2025_01_10.gz
[11/304] SKIP 2025_01_11.gz
[12/304] SKIP 2025_01_12.gz
[13/304] SKIP 2025_01_13.gz
[14/304] SKIP 2025_01_14.gz
[15/304] SKIP 2025_01_15.gz
[16/304] SKIP 2025_01_17.gz
[17/304] SKIP 2025_01_18.gz
[18/304] SKIP 2025_01_19.gz
[19/304] SKIP 2025_01_20.gz
[20/304] SKIP 2025_01_21.gz
[21/304] SKIP 2025_01_22.gz
[22/304] SKIP 2025_01_23.gz
[23/304] SKIP 2025_01_24.gz
[24/304] SKIP 2025_01_25.gz
[25/304] SKIP 2025_01_26.gz
[26/304] SKIP 2025_01_27.gz
[27/304] SKIP 2025_01_28.gz
[28/304] SKIP 2025_01_29.gz
[29/304] SKIP 2025_01_30.gz
[30/304] SKIP 2025_01_31.gz
[31/304] SKIP 2025_02_01.gz
[32/304] SKIP 2025_02_02.gz
[33/304] SKIP 2025_02_03.gz
[34/304] SKIP 2025_02_04.gz
[35/304] SKIP 2025_02_05.gz
[36/304] SKIP 2025_02_06.gz
[

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[123/304] 2025_05_04.gz → 2,617,428 rows (10.1s)
[124/304] SKIP 2025_05_05.gz
[125/304] SKIP 2025_05_06.gz


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[126/304] 2025_05_07.gz → 2,560,922 rows (12.4s)
[127/304] SKIP 2025_05_08.gz
[128/304] SKIP 2025_05_09.gz
[129/304] SKIP 2025_05_10.gz
[130/304] SKIP 2025_05_11.gz
[131/304] SKIP 2025_05_12.gz
[132/304] SKIP 2025_05_13.gz
[133/304] SKIP 2025_05_14.gz
[134/304] SKIP 2025_05_15.gz
[135/304] SKIP 2025_05_16.gz
[136/304] SKIP 2025_05_17.gz
[137/304] SKIP 2025_05_18.gz
[138/304] SKIP 2025_05_19.gz
[139/304] SKIP 2025_05_20.gz
[140/304] SKIP 2025_05_21.gz
[141/304] SKIP 2025_05_22.gz
[142/304] SKIP 2025_05_23.gz
[143/304] SKIP 2025_05_24.gz
[144/304] SKIP 2025_05_25.gz
[145/304] SKIP 2025_05_26.gz
[146/304] SKIP 2025_05_27.gz
[147/304] SKIP 2025_05_28.gz
[148/304] SKIP 2025_05_29.gz
[149/304] SKIP 2025_05_30.gz
[150/304] SKIP 2025_05_31.gz
[151/304] SKIP 2025_06_01.gz
[152/304] SKIP 2025_06_02.gz
[153/304] SKIP 2025_06_03.gz
[154/304] SKIP 2025_06_04.gz
[155/304] SKIP 2025_06_05.gz
[156/304] SKIP 2025_06_06.gz
[157/304] SKIP 2025_06_07.gz
[158/304] SKIP 2025_06_08.gz
[159/304] SKIP 2025_06_

In [5]:
## Verify — check a sample parquet file

sample_files = list(STAGE_DIR.rglob("*.parquet"))
print(f"Parquet files on disk: {len(sample_files)}")

con.execute(f"""
    SELECT *
    FROM read_parquet('{sample_files[0]}')
    LIMIT 3
""").df()

Parquet files on disk: 418


,vehicle_id,imsi,event_ts,cell_id,rat,source_file,event_date
0,F140A413CDBACB245F79DAE1D7438EC4,000000000000000,2025-01-01 00:02:01.793,69492489,4G,2025_01_01.gz,2025-01-01
1,F140A413CDBACB245F79DAE1D7438EC4,000000000000000,2025-01-01 00:10:05.626,69492489,4G,2025_01_01.gz,2025-01-01
2,F140A413CDBACB245F79DAE1D7438EC4,000000000000000,2025-01-01 00:18:09.436,69492708,4G,2025_01_01.gz,2025-01-01


In [6]:
## Register handover_events in analytics DuckDB

analytics = duckdb.connect("/home/jovyan/data/analytics.duckdb")
analytics.execute("SET memory_limit='8GB'")

analytics.execute("DROP TABLE IF EXISTS handover_events")
analytics.execute(f"""
    CREATE TABLE handover_events AS
    SELECT
        vehicle_id,
        imsi,
        event_ts,
        cell_id,
        rat,
        source_file,
        CAST(event_ts AS DATE) AS event_date
    FROM read_parquet('{STAGE_DIR}/**/*.parquet')
    WHERE vehicle_id IS NOT NULL
      AND cell_id    IS NOT NULL
      AND event_ts   IS NOT NULL
""")

analytics.execute("""
    SELECT
        COUNT(*)                   AS total_rows,
        COUNT(DISTINCT vehicle_id) AS vehicles,
        COUNT(DISTINCT cell_id)    AS cells,
        MIN(event_ts)              AS min_event,
        MAX(event_ts)              AS max_event
    FROM handover_events
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,vehicles,cells,min_event,max_event
0,913539144,99610,941090,2025-01-01 00:02:01.793,2025-11-01 23:59:56.110
